# 5. Проверка устойчивости спецификаций (многоканальный случай)

**Цель:** сопоставить несколько допустимых спецификаций `GraphEVTPipeline` на Kuramoto-ряде, включая выбор метода графа. Таблица показывает, какие выводы — обнаружение, момент тревоги, оценка источника — устойчивы к настройкам.

In [1]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_synchronized_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_synchronized_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
assert np.isfinite(x).all() and x.ndim == 2

In [2]:
BASELINE_SIZE = 300
specifications = [
    {"role": "sensitive", "tail_quantile": 0.85, "alarm_probability": 0.97, "persistence": 1, "graph_method": "ar"},
    {"role": "primary", "tail_quantile": 0.90, "alarm_probability": 0.99, "persistence": 2, "graph_method": "ar"},
    {"role": "conservative", "tail_quantile": 0.95, "alarm_probability": 0.999, "persistence": 3, "graph_method": "ar"},
    {"role": "alt_graph_dfa", "tail_quantile": 0.90, "alarm_probability": 0.99, "persistence": 2, "graph_method": "dfa"},
]
rows = []
for spec in specifications:
    evt = EVTConfig(
        baseline_size=BASELINE_SIZE,
        tail_quantile=spec["tail_quantile"],
        alarm_probability=spec["alarm_probability"],
        persistence=spec["persistence"],
        top_k=3,
    )
    run = GraphEVTPipeline(
        evt,
        graph=GraphConfig(method=spec["graph_method"], edge_threshold=0.35, random_state=SEED),
        input_config=InputConfig(missing="error", require_regular_time=True),
        verbose=True,
    ).run_detailed(x)
    rows.append({
        **spec,
        "alarm_threshold": run.detection.alarm_threshold,
        "post_baseline_exceedances": int(np.sum(run.detection.indicator[BASELINE_SIZE:] > run.detection.alarm_threshold)),
        "detected": run.detection.detected,
        "first_persistent_index": run.detection.time_index,
        "graph_available": run.graph is not None,
        "estimated_source": None if run.ranking is None else channel_names[run.ranking.source],
    })
sensitivity = pd.DataFrame(rows)
sensitivity

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

,role,tail_quantile,alarm_probability,persistence,graph_method,alarm_threshold,post_baseline_exceedances,detected,first_persistent_index,graph_available,estimated_source
0,sensitive,0.85,0.970,1,ar,5.468804,54,True,300.0,True,channel_08
1,primary,0.90,0.990,2,ar,5.494980,33,True,300.0,True,channel_08
2,conservative,0.95,0.999,3,ar,5.509289,5,False,NaN,False,NaN
3,alt_graph_dfa,0.90,0.990,2,dfa,5.494980,33,True,300.0,True,channel_08


In [3]:
artifact_dir = ROOT / "results/n-d"
artifact_dir.mkdir(parents=True, exist_ok=True)
sensitivity.to_csv(artifact_dir / "05_specification_review.csv", index=False)

agreement = {
    "all_detected": bool(sensitivity["detected"].all()),
    "none_detected": bool((~sensitivity["detected"]).all()),
    "same_first_index": bool(sensitivity["first_persistent_index"].nunique(dropna=False) == 1),
    "same_estimated_source": bool(sensitivity["estimated_source"].nunique(dropna=False) == 1),
    "library_version": version("graph-evt-agent"),
}
agreement

{'all_detected': False,
 'none_detected': False,
 'same_first_index': False,
 'same_estimated_source': False,
 'library_version': '0.1.0'}

Разные строки — прозрачные статистические спецификации, включая альтернативный метод графа (`dfa` вместо `ar`). Расхождение между ними является анализом чувствительности, а совпадение не доказывает истинность источника.

**Самопроверка:** меняет ли метод графа момент тревоги, а не только ranking? Какая спецификация здесь наиболее консервативна? Почему согласие всех строк по одному ряду всё ещё не гарантирует обобщение?